PyTorch 优化器 `torch.optim`

In [2]:
import torch
from torch import nn

In [6]:
# 自定义一个MLP，多层感知机，使用 nn.Sequential 序列化定义
class MLP(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super(MLP, self).__init__()
        self.sequential = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, int(hidden_dim / 2)),
            nn.ReLU(),
            nn.Linear(int(hidden_dim / 2), out_features),
        )

    def forward(self, x):
        return self.sequential(x)

model = MLP(28 * 28, 256, 10)


In [5]:
## 1. 优化器在训练流程中的位置

# optimizer.zero_grad(set_to_none=True) # 清空上一轮梯度并设置梯度为None，并设置优化器减少不必要的内存开销
# outputs = model(inputs)               # 训练数据经过模型前向传播，计算并得到预测结果
# loss = criterion(outputs, targets)    # 模型输出与真实标签计算损失，衡量模型预测的误差
# loss.backward()                       # 梯度反向传播。将梯度累积到相关叶子张量的 .grad 中，通常包括模型参数
# optimizer.step()                      # 更新参数。根据 .grad 中的梯度，按照优化器的更新规则修改模型参数

# `nn.Module` 定义参数和前向计算，损失函数产生优化目标，autograd 计算参数 `.grad`，optimizer 读取梯度并更新参数，scheduler 修改 optimizer 的超参数。优化器不会自动计算损失，也不会自动调用 `backward()`。


In [39]:
## 2. 梯度下降基础
# w = w - eta * y
# `eta` 是学习率。梯度指向损失上升最快的方向，因此减去梯度可以降低局部损失。

# 通过梯度下降计算 loss=(x-2)^2的最小值
wlist = []
losss = []
w = torch.tensor(5.0, requires_grad=True)
for _ in range(50):
    loss = (w - 2) ** 2
    losss.append(loss.detach().item())
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
        wlist.append(w.detach().numpy())
    w.grad = None


In [43]:
# backward
x = torch.tensor(5, dtype=torch.float64, requires_grad=True)
(3*x -2).backward()
print(x.grad) # 输出3
(3*x -2).backward()
print(x.grad) # 输出6=3+3 第一个3是上一次求梯度保留下来的
# 因此也侧面证明了，每一轮训练开始的时候都必须清空上一轮保留下来的梯度，使用 optimizer.zero_grad(set_to_none=True) 来清空每一轮保留的梯度

tensor(3., dtype=torch.float64)
tensor(6., dtype=torch.float64)


In [56]:
## 3. 第一个优化器
model = nn.Linear(in_features=1, out_features=1, bias=True)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
# y = 2x+1
# 第一维是每一条训练数据，第二维是每一条训练数据的具体特征数据
x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]], requires_grad=True)
y = torch.tensor([[3.0], [5.0], [7.0], [9.0], [11.0]])

# 训练
for _ in range(1000):
    # 清空上一轮梯度
    optimizer.zero_grad(set_to_none=True)
    # 前向传播
    output = model(x)
    # 计算损失
    loss = criterion(output, y)
    # 损失反向传播
    loss.backward()
    # 更新参数
    optimizer.step()

for name, param in model.named_parameters():
    print(name, param)


weight Parameter containing:
tensor([[2.0086]], requires_grad=True)
bias Parameter containing:
tensor([0.9690], requires_grad=True)


In [58]:
# Momentum 累积历史方向，减少震荡：
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, nesterov=True
)
# SGD 简单、额外状态少，但对学习率敏感。图像分类中，SGD + momentum 配合较长训练和学习率调度常有很好的最终泛化表现。

In [59]:
## 7. Adam 与 AdamW

# Adam 通常容易作为新任务的快速基线，但每个参数有额外状态，显存占用高于 SGD。
# AdamW 将权重衰减与自适应梯度更新解耦：

optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-2
)

# Transformer、语言模型和现代视觉模型常以 AdamW 作为起点。经验上常对 bias 和归一化层不做衰减：



In [67]:
## 8. 其他常见优化器
# - RMSprop 按梯度平方移动平均缩放更新，历史上常见于循环网络。
# - Adagrad 适合稀疏特征，但累计平方梯度可能使学习率过早变小。
# - Adadelta 改进了 Adagrad，如今使用相对少。
# - LBFGS 是准二阶方法，需要闭包重新计算 loss，适合小规模平滑问题。
rmsprop = torch.optim.RMSprop(model.parameters(), lr=1e-3)
adagrad = torch.optim.Adagrad(model.parameters(), lr=1e-2)

le = 1e-5

print(le*1)
print(le * 10)
print(le * 100)

1e-05
0.0001
0.001


In [69]:
# torch.optim.lr_scheduler 学习率调度器
# 不使用 torch.optim.lr_scheduler，也完全可以正常训练模型。 它不是训练模型的必需组件，而是一个用于动态调整学习率（Learning Rate）的工具。
# 你可以先记住这三个概念：
# - optimizer：决定如何根据梯度更新模型参数。
# - lr：控制每次更新参数时，步子迈多大。
# - lr_scheduler：根据训练进度或训练效果，自动调整这个步子的大小。
# 一开始的时候初始学习率lr可以设置大一点（梯度下降的步子大一些），后续epoch的时候吧学习率lr调的小一点，这样有益于收敛

### 10.1 `StepLR`、`MultiStepLR`、`CosineAnnealingLR`
# StepLR 的作用是：每隔指定数量的训练周期，就把学习率乘以一个系数。
# 每隔10个epoch调整一次，new_epoch = current_cpoch * 0.1
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# MultiStepLR 在指定的 Epoch 节点降低学习率	已知关键训练阶段
multi_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 60, 90], gamma=0.1)
# CosineAnnealingLR 按余弦曲线逐步调整学习率	常规深度学习训练
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

# ReduceLROnPlateau
# 有时候，你不希望每隔固定的 Epoch 就降低学习率，而是希望：当验证集损失连续几轮没有明显改善时，再降低学习率。
ReduceLROnPlateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",       # 监控的指标越小越好
    factor=0.5,       # 学习率乘以 0.5
    patience=2        # 连续 2 个 Epoch 未改善后触发调整
)


# lr_scheduler 的使用方式
# 通常先执行 `optimizer.step()` 再执行 `scheduler.step()`。`T_max` 与 scheduler 的调用次数对应；每 batch 调用就按 step 理解，每 epoch 调用就按 epoch 理解。
# for epoch in range(30):
#     train_one_epoch(model, optimizer) # 实际的每一轮训练过程
#     scheduler.step()
#     # 当使用的是 ReduceLROnPlateau 调节器的时候必须使用验证集来计算损失，如何将损失传入调节器
#     valid_loss = validate(model)
#     ReduceLROnPlateau_scheduler.step(valid_loss)  # 必须传入监控指标

